# CNN 미니 프로젝트 — base.ipynb

전이학습 없이 처음부터 학습하는 기본 CNN · Colab (T4 GPU) · PyTorch

1. *런타임 → 런타임 유형 변경 → T4 GPU* 선택 후 위에서부터 차례로 실행합니다.
2. 데이터셋이 없으면 `DATA_MODE = "cifar10"` 그대로 두고 실행합니다.
3. 우리 데이터셋이 준비되면 `DATA_MODE = "folder"` 로 바꾸고 `DATA_DIR` 만 고칩니다.
4. 각 층의 역할은 **README.md** 를 참고하세요.

## 1. 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

DATA_MODE  = "cifar10"                              # "cifar10" | "folder"
DATA_DIR   = "/content/drive/MyDrive/our_dataset"   # folder 모드: train/<클래스>/*.jpg, test/<클래스>/*.jpg
IMG_SIZE   = 32
BATCH_SIZE = 128
EPOCHS     = 10
LR         = 1e-3

torch.manual_seed(0)                                # 다 같이 같은 결과가 나오도록 고정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. 데이터

In [ ]:
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                              # 0~255 → 0~1, (H, W, C) → (C, H, W)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),   # → -1~1
])

if DATA_MODE == "cifar10":
    train_set = datasets.CIFAR10("data", train=True,  download=True, transform=tf)
    test_set  = datasets.CIFAR10("data", train=False, download=True, transform=tf)
else:
    if DATA_DIR.startswith("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
    train_set = datasets.ImageFolder(f"{DATA_DIR}/train", transform=tf)
    test_set  = datasets.ImageFolder(f"{DATA_DIR}/test",  transform=tf)

CLASSES = train_set.classes                         # 폴더 이름 순서 = 레이블 번호
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)
print(f"클래스 {len(CLASSES)}개 | train {len(train_set)} | test {len(test_set)} | 한 장 {tuple(train_set[0][0].shape)}")

## 3. 모델

| 코드 | 하는 일 |
|---|---|
| `Conv2d(..., kernel_size=3, padding=1)` | 3×3 필터로 무늬 찾기 (padding=1 → 크기 유지) |
| `MaxPool2d(2)` | 2×2 마다 최댓값 → 크기 절반 |
| `Flatten()` | (채널, 높이, 너비) → 1차원 |
| `Linear(..., 클래스 수)` | 클래스별 점수. Keras 의 `Dense(10, activation="softmax")` 와 같은 자리 |

> softmax 는 모델에 넣지 않습니다. 학습 때 `CrossEntropyLoss` 가 내부에서 계산합니다.

In [ ]:
class BaseCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.flatten = nn.Flatten()
        n_flat = 128 * (IMG_SIZE // 8) ** 2             # MaxPool 3번 → 크기 1/8
        self.classifier = nn.Sequential(
            nn.Linear(n_flat, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.flatten(self.features(x)))

model = BaseCNN(len(CLASSES)).to(device)
print(f"파라미터 {sum(p.numel() for p in model.parameters()):,}개")   # CIFAR-10 기준 620,362

## 4. 층마다 shape 확인

In [ ]:
x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
print("입력".ljust(12), tuple(x.shape[1:]))
for layer in [*model.features, model.flatten, *model.classifier]:
    x = layer(x)
    print(type(layer).__name__.ljust(12), tuple(x.shape[1:]))

## 5. 학습

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(1) == labels).sum().item()
    return correct / len(loader.dataset)

history = []
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)            # 1) 예측
        loss = loss_fn(outputs, labels)    # 2) 오차 계산
        optimizer.zero_grad()              # 3) 이전 기울기 지우기
        loss.backward()                    # 4) 기울기 계산
        optimizer.step()                   # 5) 가중치 갱신
    acc = evaluate(model, test_loader)
    history.append(acc)
    print(f"epoch {epoch + 1:2d}/{EPOCHS}  loss {loss.item():.4f}  test acc {acc:.4f}")

## 6. 결과

In [ ]:
plt.plot(range(1, EPOCHS + 1), history, "o-")
plt.xlabel("epoch"); plt.ylabel("test accuracy"); plt.grid(alpha=.3); plt.show()
print(f"최종 test 정확도 {history[-1]:.4f}")

# 테스트 이미지 한 장 예측 — softmax 로 점수를 확률로 바꿔 봅니다
image, label = test_set[0]
model.eval()
with torch.no_grad():
    prob = F.softmax(model(image.unsqueeze(0).to(device)), dim=1)[0].cpu()
plt.imshow((image.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1)); plt.axis("off")
plt.title(f"true: {CLASSES[label]} / pred: {CLASSES[prob.argmax()]} ({prob.max():.0%})"); plt.show()